In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("=" * 55)
print("SLIIT Feature Engineering")
print("R26-IT-059 | IT22215710 | Karunarathne D C")
print("=" * 55)
print("\nLibraries imported successfully!")

SLIIT Feature Engineering
R26-IT-059 | IT22215710 | Karunarathne D C

Libraries imported successfully!


In [2]:
# Load SLIIT synthetic data
DATA_PATH = "../data/raw/"

sliit_df = pd.read_csv(DATA_PATH + "sliit_lms_data.csv")
student_info = pd.read_csv(DATA_PATH + "sliit_student_info.csv")
modules_df = pd.read_csv(DATA_PATH + "sliit_modules.csv")
calendar_df = pd.read_csv(DATA_PATH + "sliit_academic_calendar.csv")

print("Data loaded successfully!")
print(f"\nLMS data shape: {sliit_df.shape}")
print(f"Students: {sliit_df['student_id'].nunique()}")
print(f"Modules: {sliit_df['module_code'].nunique()}")
print(f"Weeks: {sliit_df['week'].nunique()}")
print(f"\nColumns: {list(sliit_df.columns)}")
print(f"\nAt-risk rate: {sliit_df.drop_duplicates('student_id')['at_risk'].mean()*100:.1f}%")

Data loaded successfully!

LMS data shape: (37500, 17)
Students: 500
Modules: 5
Weeks: 15

Columns: ['student_id', 'student_name', 'student_type', 'module_code', 'module_name', 'week', 'week_type', 'weekly_clicks', 'active_days', 'forum_posts', 'resource_views', 'quiz_attempts', 'submission_latency', 'lecture_video_views', 'content_diversity', 'late_submission', 'at_risk']

At-risk rate: 35.0%


In [3]:
# Build curriculum baseline using top 25% engaged students
print("Building curriculum baseline...")

# Calculate total engagement per student
total_engagement = sliit_df.groupby('student_id')[
    'weekly_clicks'
].sum().reset_index()
total_engagement.columns = ['student_id', 'total_clicks']

# Get 75th percentile threshold
threshold = total_engagement['total_clicks'].quantile(0.75)
engaged_students = total_engagement[
    total_engagement['total_clicks'] >= threshold
]['student_id'].tolist()

print(f"Highly engaged students identified: {len(engaged_students)}")

# Filter to engaged students data
engaged_data = sliit_df[
    sliit_df['student_id'].isin(engaged_students)
]

# Calculate expected behavior per module per week
expected_behavior = engaged_data.groupby(
    ['module_code', 'week']
).agg(
    expected_clicks=('weekly_clicks', 'mean'),
    expected_active_days=('active_days', 'mean'),
    expected_resources=('resource_views', 'mean')
).reset_index().round(1)

print(f"\nExpected behavior calculated!")
print(f"\nSample — Module IT3080:")
print(expected_behavior[
    expected_behavior['module_code'] == 'IT3080'
].head(10).to_string(index=False))

# Save
expected_behavior.to_csv(
    "../results/metrics/sliit_expected_behavior.csv",
    index=False
)
print(f"\nSaved to results/metrics/sliit_expected_behavior.csv")

Building curriculum baseline...
Highly engaged students identified: 125

Expected behavior calculated!

Sample — Module IT3080:
module_code  week  expected_clicks  expected_active_days  expected_resources
     IT3080     1             60.5                   2.6                 7.2
     IT3080     2            119.4                   4.5                20.2
     IT3080     3            122.2                   4.5                19.8
     IT3080     4            120.5                   4.5                19.2
     IT3080     5            124.1                   4.5                19.4
     IT3080     6            120.0                   4.4                19.3
     IT3080     7            123.0                   4.4                19.5
     IT3080     8            123.0                   4.5                19.3
     IT3080     9            201.3                   6.5                38.9
     IT3080    10            121.2                   4.5                19.4

Saved to results/metrics

In [4]:
# Calculate curriculum compliance per student per week
print("Calculating curriculum compliance scores...")

# Merge with expected behavior
sliit_df = sliit_df.merge(
    expected_behavior,
    on=['module_code', 'week'],
    how='left'
)

# Calculate compliance ratios
sliit_df['click_compliance'] = (
    sliit_df['weekly_clicks'] / 
    (sliit_df['expected_clicks'] + 1)
).clip(upper=2.0)

sliit_df['day_compliance'] = (
    sliit_df['active_days'] / 
    (sliit_df['expected_active_days'] + 1)
).clip(upper=2.0)

sliit_df['curriculum_compliance'] = (
    sliit_df['click_compliance'] * 0.5 +
    sliit_df['day_compliance'] * 0.5
)

print(f"Compliance scores calculated!")
print(f"\nMean compliance by student type:")
print(sliit_df.groupby('student_type')[
    'curriculum_compliance'
].mean().round(4))

print(f"\nMean compliance by at_risk:")
print(sliit_df.groupby('at_risk')[
    'curriculum_compliance'
].mean().round(4))

Calculating curriculum compliance scores...
Compliance scores calculated!

Mean compliance by student type:
student_type
BURNOUT         0.4424
LATE_BURNOUT    0.7547
NORMAL          0.9022
STRUGGLING      0.2535
Name: curriculum_compliance, dtype: float64

Mean compliance by at_risk:
at_risk
0    0.7525
1    0.5763
Name: curriculum_compliance, dtype: float64


In [5]:
# Finalize 10 features for VAE training
print("Finalizing feature set...")

# Add baseline flag
sliit_df['is_baseline'] = sliit_df['week'].isin([2, 3]).astype(int)

# Define final 10 features
FEATURE_COLS = [
    'weekly_clicks',
    'active_days',
    'content_diversity',
    'forum_posts',
    'resource_views',
    'quiz_attempts',
    'submission_latency',
    'lecture_video_views',
    'curriculum_compliance',
    'click_compliance'
]

print(f"Final 10 features:")
for i, f in enumerate(FEATURE_COLS):
    print(f"  F{i+1}: {f}")

# Filter out orientation week (week 1) for cleaner baseline
features_final = sliit_df[sliit_df['week'] >= 2].copy()

print(f"\nFinal dataset shape: {features_final.shape}")
print(f"Students: {features_final['student_id'].nunique()}")
print(f"Weeks: {sorted(features_final['week'].unique())}")

# Save final feature set
features_final.to_csv(
    "../results/metrics/sliit_features_final.csv",
    index=False
)

print(f"\nSaved to results/metrics/sliit_features_final.csv")
print(f"\nReady for VAE training!")
print(f"Expected AUC improvement over OULAD (0.6039)")
print(f"due to clearer behavioral compliance signal (0.176 vs 0.155)")

Finalizing feature set...
Final 10 features:
  F1: weekly_clicks
  F2: active_days
  F3: content_diversity
  F4: forum_posts
  F5: resource_views
  F6: quiz_attempts
  F7: submission_latency
  F8: lecture_video_views
  F9: curriculum_compliance
  F10: click_compliance

Final dataset shape: (35000, 24)
Students: 500
Weeks: [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]

Saved to results/metrics/sliit_features_final.csv

Ready for VAE training!
Expected AUC improvement over OULAD (0.6039)
due to clearer behavioral compliance signal (0.176 vs 0.155)
